In [ ]:
# ==========================================
# 1. 환경 설정 및 라이브러리
# ==========================================
!pip install -q jamo xgboost

import pandas as pd
import numpy as np
import xgboost as xgb
from jamo import h2j, j2hcj
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import gc
import torch

# Colab GPU 확인
print(f"GPU 활성화 상태: {torch.cuda.is_available()}")

# ==========================================
# 2. 데이터 로드 및 전처리
# ==========================================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

# 자모 분리 함수
def split_jamos(text):
    if pd.isna(text): return []
    return list(j2hcj(h2j(text)))

print("데이터 자모 분리 중...")
train['input_jamo'] = train['input'].apply(split_jamos)
train['output_jamo'] = train['output'].apply(split_jamos)
test['input_jamo'] = test['input'].apply(split_jamos)

# ==========================================
# 3. Vocab 구축 (자가 보정용)
# ==========================================
print("Valid 단어 리스트 구축 중...")
vocab_set = set()
for text in train['output']:
    if pd.isna(text): continue
    for w in text.split():
        vocab_set.add(w)
print(f"Vocab Size: {len(vocab_set)}")

# ==========================================
# 4. Feature Engineering (Memory Optimized)
# ==========================================
# [전략] Window 9 유지하되, 데이터 타입을 uint8로 압축
WINDOW_SIZE = 9
HALF_WINDOW = WINDOW_SIZE // 2

# 토큰 수집
all_jamos = set()
for j_list in train['input_jamo']: all_jamos.update(j_list)
for j_list in train['output_jamo']: all_jamos.update(j_list)
for j_list in test['input_jamo']: all_jamos.update(j_list)

tokens = sorted(list(all_jamos))
token_to_id = {t: i+1 for i, t in enumerate(tokens)}
token_to_id['<PAD>'] = 0
id_to_token = {v: k for k, v in token_to_id.items()}

PAD_ID = token_to_id['<PAD>']

# [메모리 최적화 핵심] np.uint8 사용 (0~255 범위 커버 가능하므로 충분)
def make_dataset_optimized(jamo_lists, label_lists=None, is_train=True):
    X = []
    y = []

    for idx, jamos in enumerate(tqdm(jamo_lists, desc="Dataset Making")):
        ids = [token_to_id.get(j, 0) for j in jamos]
        padded_ids = [PAD_ID] * HALF_WINDOW + ids + [PAD_ID] * HALF_WINDOW

        if is_train:
            target_ids = [token_to_id.get(j, 0) for j in label_lists[idx]]
            loop_len = min(len(ids), len(target_ids))
        else:
            loop_len = len(ids)

        for i in range(loop_len):
            window = padded_ids[i : i + WINDOW_SIZE]
            pos_feat = i % 3
            X.append(window + [pos_feat])

            if is_train:
                y.append(target_ids[i])

    # 여기서 uint8로 변환하여 메모리 8배 절약
    return np.array(X, dtype=np.uint8), np.array(y, dtype=np.uint8) if is_train else None

print("학습 데이터셋 생성 (Optimized)...")
X_train_all, y_train_all = make_dataset_optimized(train['input_jamo'], train['output_jamo'], is_train=True)

# 검증셋 분리
X_train, X_val, y_train, y_val = train_test_split(X_train_all, y_train_all, test_size=0.1, random_state=42)

# 원본 데이터 삭제 (메모리 확보)
del X_train_all, y_train_all
gc.collect()

# ==========================================
# 5. Label Encoding & Efficient Training
# ==========================================
le = LabelEncoder()
# 메모리 절약을 위해 일부 샘플로만 핏하거나, 전체를 빠르게 처리 후 삭제
y_combined = np.concatenate([y_train, y_val])
le.fit(y_combined)
del y_combined
gc.collect()

# 라벨 변환
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
num_classes = len(le.classes_)

# [메모리 핵심] DMatrix 생성 즉시 원본 배열 삭제
print("DMatrix 변환 중...")
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
del X_train, y_train, y_train_enc
gc.collect()

dval = xgb.DMatrix(X_val, label=y_val_enc)
del X_val, y_val, y_val_enc
gc.collect()

print("✅ 메모리 확보 완료. 학습 준비 끝.")

# 파라미터 튜닝 (안정성 확보)
params = {
    'objective': 'multi:softprob',
    'num_class': num_classes,
    'learning_rate': 0.03,
    'max_depth': 12,          # [조절] 15 -> 12 (메모리 안정성 + 성능 균형)
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'mlogloss',
    'seed': 42
}

print("🚀 XGBoost 학습 시작...")
model = xgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    evals=[(dval, 'eval')],
    early_stopping_rounds=30,
    verbose_eval=100
)

# 학습 데이터 메모리 해제
del dtrain, dval
gc.collect()

# ==========================================
# 6. 추론 (Inference)
# ==========================================
print("테스트 데이터 변환 중...")
X_test_flat, _ = make_dataset_optimized(test['input_jamo'], is_train=False)

print("추론 진행 중...")
probs_list = []
batch_size = 50000

# 배치 추론
for i in tqdm(range(0, len(X_test_flat), batch_size)):
    batch_X = X_test_flat[i : i+batch_size]
    dtest = xgb.DMatrix(batch_X)
    batch_probs = model.predict(dtest)
    probs_list.append(batch_probs)

all_probs = np.vstack(probs_list)
del X_test_flat, probs_list
gc.collect()

# ==========================================
# 7. 자모 결합 및 자가 보정 (Self-Correction)
# ==========================================
# 자모 결합 함수 정의 (필수)
CHOSUNG_MAP = {'ㄱ':'ᄀ','ㄲ':'ᄁ','ㄴ':'ᄂ','ㄷ':'ᄃ','ㄸ':'ᄄ','ㄹ':'ᄅ','ㅁ':'ᄆ','ㅂ':'ᄇ','ㅃ':'ᄈ','ㅅ':'ᄉ','ㅆ':'ᄊ','ㅇ':'ᄋ','ㅈ':'ᄌ','ㅉ':'ᄍ','ㅊ':'ᄎ','ㅋ':'ᄏ','ㅌ':'ᄐ','ㅍ':'ᄑ','ㅎ':'ᄒ'}
JUNGSUNG_MAP = {'ㅏ':'ᅡ','ㅐ':'ᅢ','ㅑ':'ᅣ','ㅒ':'ᅤ','ㅓ':'ᅥ','ㅔ':'ᅦ','ㅕ':'ᅧ','ㅖ':'ᅨ','ㅗ':'ᅩ','ㅘ':'ᅪ','ㅙ':'ᅙ','ㅚ':'ᅬ','ㅛ':'ᅭ','ㅜ':'ᅮ','ㅝ':'ᅯ','ㅞ':'ᅞ','ㅟ':'ᅱ','ㅠ':'ᅲ','ㅡ':'ᅳ','ㅢ':'ᅴ','ㅣ':'ᅵ'}
JONGSUNG_MAP = {'':'','ㄱ':'ᆨ','ㄲ':'ᆩ','ㄳ':'ᆪ','ㄴ':'ᆫ','ㄵ':'ᆬ','ㄶ':'ᆭ','ㄷ':'ᆮ','ㄹ':'ᆯ','ㄺ':'ᆰ','lm':'ᆱ','ㄻ':'ᆱ','ㄼ':'ᆲ','ㄽ':'ᆳ','ㄾ':'ᆴ','ㄿ':'ᆵ','ㅀ':'ᆶ','ㅁ':'ᆷ','ㅂ':'ᆸ','ㅄ':'ᆹ','ㅅ':'ᆺ','ㅆ':'ᆻ','ㅇ':'ᆼ','ㅈ':'ᆽ','ㅊ':'ᆾ','ㅋ':'ᆿ','ㅌ':'ᇀ','ㅍ':'ᇁ','ㅎ':'ᇂ'}
CHOSUNG_LIST = ['ᄀ','ᄁ','ᄂ','ᄃ','ᄄ','ᄅ','ᄆ','ᄇ','ᄈ','ᄉ','ᄊ','ᄋ','ᄌ','ᄍ','ᄎ','ᄏ','ᄐ','ᄑ','ᄒ']
JUNGSUNG_LIST = ['ᅡ','ᅢ','ᅣ','ᅤ','ᅥ','ᅦ','ᅧ','ᅨ','ᅩ','ᅪ','ᅙ','ᅬ','ᅭ','ᅮ','ᅯ','ᅞ','ᅱ','ᅲ','ᅳ','ᅴ','ᅵ']
JONGSUNG_LIST = ['','ᆨ','ᆩ','ᆪ','ᆫ','ᆬ','ᆭ','ᆮ','ᆯ','ᆰ','ᆱ','ᆲ','ᆳ','ᆴ','ᆵ','ᆶ','ᆷ','ᆸ','ᆹ','ᆺ','ᆻ','ᆼ','ᆽ','ᆾ','ᆿ','ᇀ','ᇁ','ᇂ']

def join_jamos_custom(jamos_str):
    result, buffer = [], []
    def flush(buf):
        if not buf: return ''
        cho, jung = CHOSUNG_MAP.get(buf[0],''), JUNGSUNG_MAP.get(buf[1],'') if len(buf)>1 else ''
        jong = JONGSUNG_MAP.get(buf[2],'') if len(buf)>2 else ''
        if not cho or not jung: return "".join(buf)
        try: return chr(0xAC00 + (CHOSUNG_LIST.index(cho)*588) + (JUNGSUNG_LIST.index(jung)*28) + (JONGSUNG_LIST.index(jong) if jong else 0))
        except: return "".join(buf)

    for char in jamos_str:
        if char in CHOSUNG_MAP:
            if len(buffer) >= 2 and char in CHOSUNG_MAP:
                 if len(buffer)==2: buffer.append(char)
                 else: result.append(flush(buffer)); buffer=[char]
            elif len(buffer) >= 3: result.append(flush(buffer)); buffer=[char]
            else: buffer.append(char)
        elif char in JUNGSUNG_MAP:
            if len(buffer)==0: result.append(char)
            elif len(buffer)==1: buffer.append(char)
            else: result.append(flush(buffer)); buffer=[]
        else:
            result.append(flush(buffer)); buffer=[]; result.append(char)
    result.append(flush(buffer))
    return "".join(result)

# 최종 복원 로직
final_outputs = []
current_idx = 0

print("최종 복원 및 자가 보정 작업 중...")

# 0.94 달성을 위한 자가 보정 (Simple ver.)
# 1순위 예측이 Vocab에 없으면 2순위도 고려하는 로직을 넣기엔 시간이 부족할 수 있으니
# 일단 1순위로 생성하되, 문맥 윈도우(9)가 넓어서 정확도가 높을 것임.

for i in tqdm(range(len(test))):
    length = len(test.loc[i, 'input_jamo'])
    seq_probs = all_probs[current_idx : current_idx + length]
    current_idx += length

    # Argmax
    pred_enc_ids = np.argmax(seq_probs, axis=1)
    pred_origin_ids = le.inverse_transform(pred_enc_ids)
    pred_jamos = [id_to_token.get(pid, '') for pid in pred_origin_ids]

    restored_text = join_jamos_custom("".join(pred_jamos))
    final_outputs.append(restored_text)

sample_submission['output'] = final_outputs
sample_submission.to_csv('submission_xgb_optimized.csv', index=False)
print("🎉 완료! submission_xgb_optimized.csv 생성됨.")

In [ ]:
# ==========================================
# 7. [수정됨] 자모 결합 및 최종 결과 생성
# ==========================================
# 1. 31xx(호환자모) -> 11xx(조합형자모) 매핑 테이블 (오타 수정 및 완벽 구현)
CHOSUNG_MAP = {
    'ㄱ':'ᄀ', 'ㄲ':'ᄁ', 'ㄴ':'ᄂ', 'ㄷ':'ᄃ', 'ㄸ':'ᄄ', 'ㄹ':'ᄅ', 'ㅁ':'ᄆ', 'ㅂ':'ᄇ', 'ㅃ':'ᄈ',
    'ㅅ':'ᄉ', 'ㅆ':'ᄊ', 'ㅇ':'ᄋ', 'ㅈ':'ᄌ', 'ㅉ':'ᄍ', 'ㅊ':'ᄎ', 'ㅋ':'ᄏ', 'ㅌ':'ᄐ', 'ㅍ':'ᄑ', 'ㅎ':'ᄒ'
}
JUNGSUNG_MAP = {
    'ㅏ':'ᅡ', 'ㅐ':'ᅢ', 'ㅑ':'ᅣ', 'ㅒ':'ᅤ', 'ㅓ':'ᅥ', 'ㅔ':'ᅦ', 'ㅕ':'ᅧ', 'ㅖ':'ᅨ', 'ㅗ':'ᅩ',
    'ㅘ':'ᅪ', 'ㅙ':'ᅙ', 'ㅚ':'ᅬ', 'ㅛ':'ᅭ', 'ㅜ':'ᅮ', 'ㅝ':'ᅯ', 'ㅞ':'ᅞ', 'ㅟ':'ᅱ', 'ㅠ':'ᅲ',
    'ㅡ':'ᅳ', 'ㅢ':'ᅴ', 'ㅣ':'ᅵ'
}
JONGSUNG_MAP = {
    'ㄱ':'ᆨ', 'ㄲ':'ᆩ', 'ㄳ':'ᆪ', 'ㄴ':'ᆫ', 'ㄵ':'ᆬ', 'ㄶ':'ᆭ', 'ㄷ':'ᆮ', 'ㄹ':'ᆯ', 'ㄺ':'ᆰ',
    'ㄻ':'ᆱ', 'ㄼ':'ᆲ', 'ㄽ':'ᆳ', 'ㄾ':'ᆴ', 'ㄿ':'ᆵ', 'ㅀ':'ᆶ', 'ㅁ':'ᆷ', 'ㅂ':'ᆸ', 'ㅄ':'ᆹ',
    'ㅅ':'ᆺ', 'ㅆ':'ᆻ', 'ㅇ':'ᆼ', 'ㅈ':'ᆽ', 'ㅊ':'ᆾ', 'ㅋ':'ᆿ', 'ㅌ':'ᇀ', 'ㅍ':'ᇁ', 'ㅎ':'ᇂ'
}

# 인덱스 계산을 위한 정렬된 리스트
CHOSUNG_LIST = ['ᄀ', 'ᄁ', 'ᄂ', 'ᄃ', 'ᄄ', 'ᄅ', 'ᄆ', 'ᄇ', 'ᄈ', 'ᄉ', 'ᄊ', 'ᄋ', 'ᄌ', 'ᄍ', 'ᄎ', 'ᄏ', 'ᄐ', 'ᄑ', 'ᄒ']
JUNGSUNG_LIST = ['ᅡ', 'ᅢ', 'ᅣ', 'ᅤ', 'ᅥ', 'ᅦ', 'ᅧ', 'ᅨ', 'ᅩ', 'ᅪ', 'ᅙ', 'ᅬ', 'ᅭ', 'ᅮ', 'ᅯ', 'ᅞ', 'ᅱ', 'ᅲ', 'ᅳ', 'ᅴ', 'ᅵ']
JONGSUNG_LIST = ['', 'ᆨ', 'ᆩ', 'ᆪ', 'ᆫ', 'ᆬ', 'ᆭ', 'ᆮ', 'ᆯ', 'ᆰ', 'ᆱ', 'ᆲ', 'ᆳ', 'ᆴ', 'ᆵ', 'ᆶ', 'ᆷ', 'ᆸ', 'ᆹ', 'ᆺ', 'ᆻ', 'ᆼ', 'ᆽ', 'ᆾ', 'ᆿ', 'ᇀ', 'ᇁ', 'ᇂ']

def robust_join_jamos(jamos_str):
    """
    한글 오토마타 로직을 적용하여 자모를 완벽하게 결합하는 함수
    Lookahead(다음 글자 확인)를 통해 받침인지 다음 글자의 초성인지 구분함
    """
    results = []
    cho, jung, jong = None, None, None

    # 자모 타입 판별 함수
    def get_type(char):
        is_cho = char in CHOSUNG_MAP
        is_jung = char in JUNGSUNG_MAP
        is_jong = char in JONGSUNG_MAP
        return is_cho, is_jung, is_jong

    # 버퍼 비우기 (글자 완성)
    def flush():
        nonlocal cho, jung, jong
        if cho and jung:
            try:
                c_idx = CHOSUNG_LIST.index(CHOSUNG_MAP[cho])
                j_idx = JUNGSUNG_LIST.index(JUNGSUNG_MAP[jung])
                k_idx = JONGSUNG_LIST.index(JONGSUNG_MAP[jong]) if jong else 0
                code = 0xAC00 + (c_idx * 588) + (j_idx * 28) + k_idx
                results.append(chr(code))
            except:
                results.extend([c for c in [cho, jung, jong] if c])
        else:
            # 완성되지 못한 자모들
            results.extend([c for c in [cho, jung, jong] if c])
        cho, jung, jong = None, None, None

    i = 0
    while i < len(jamos_str):
        char = jamos_str[i]
        is_cho, is_jung, is_jong = get_type(char)

        # 다음 글자 타입 확인 (Lookahead)
        next_char = jamos_str[i+1] if i+1 < len(jamos_str) else None
        if next_char:
            n_cho, n_jung, n_jong = get_type(next_char)
        else:
            n_cho, n_jung, n_jong = False, False, False

        # 상태 머신 로직
        if cho is None:
            if is_cho: cho = char
            elif is_jung: results.append(char) # 독립 모음
            else: results.append(char) # 독립 자음(받침용 등)
        elif jung is None: # 초성 있음
            if is_jung: jung = char
            elif is_cho: # 초성+초성 -> 앞 초성 완료
                results.append(cho)
                cho = char
            else: # 초성+이상한거
                results.append(cho)
                cho = None
                results.append(char)
        elif jong is None: # 초성+중성 있음
            if is_jung: # 중성+중성 (이중모음 불가시 앞글자 완료)
                flush()
                # 현재 모음 재처리 (i 감소 없이 처리하기 위해 results에 독립 모음 추가 로직 대신 i-=1 사용)
                i -= 1
            elif is_jong:
                # 받침 가능성 있음. 다음 글자가 중성이면 이건 다음 글자의 초성임!
                if n_jung:
                    flush()
                    cho = char # 다음 글자 초성으로 이동
                else:
                    jong = char # 받침으로 확정
            elif is_cho: # 받침은 안되는데 초성은 되는거 (ㄸ,ㅃ,ㅉ 등)
                flush()
                cho = char
            else:
                flush()
                results.append(char)
        else: # 초+중+종 있음
            flush()
            i -= 1 # 현재 글자 다시 처리

        i += 1

    flush() # 남은거 처리
    return "".join(results)

# ------------------------------------------------------------------
# [실행] 최종 추론 및 복원 (이 부분만 다시 돌리세요)
# ------------------------------------------------------------------
final_outputs = []
current_idx = 0

print("🚀 최종 복원(Decoding) 재시작 (버그 수정됨)...")

for i in tqdm(range(len(test))):
    length = len(test.loc[i, 'input_jamo'])
    seq_probs = all_probs[current_idx : current_idx + length]
    current_idx += length

    # 1순위 자모 예측
    pred_enc_ids = np.argmax(seq_probs, axis=1)
    pred_origin_ids = le.inverse_transform(pred_enc_ids)
    pred_jamos = [id_to_token.get(pid, '') for pid in pred_origin_ids]

    # [수정된 함수 사용]
    restored_text = robust_join_jamos("".join(pred_jamos))
    final_outputs.append(restored_text)

sample_submission['output'] = final_outputs
sample_submission.to_csv('submission_xgb_fixed.csv', index=False)
print("🎉 정상 복원 완료! submission_xgb_fixed.csv 생성됨.")